In [1]:
from dlinvc.hyperism_scene_render import hyperism_scene_render
from dlinvc.agents import CoarseGeometryInspectorAgent
from pathlib import Path
from time import sleep

In [2]:
hyperism_dir = Path("../hyperism")
metadata_file = hyperism_dir / "metadata_camera_parameters.csv"
output_dir = Path("../outputs")

scenes = sorted([d for d in hyperism_dir.iterdir() if d.is_dir() and d.name.startswith("ai_")])

print(f"Found {len(scenes)} scenes")

for scene_dir in scenes:
    scene_name = scene_dir.name
    scene, img = hyperism_scene_render(
        scene_dir=str(scene_dir), frame_id=0, hyperism_cam_metadata_file=str(metadata_file), output_dir="../outputs"
    )

Found 30 scenes


In [4]:
inspector = CoarseGeometryInspectorAgent()

with open("../prompts/scene_inspector_prompt.txt") as f:
    prompt = f.read()

scores = {}

for scene_dir in scenes:
    scene_name = scene_dir.name

    print("Evaluating:", scene_name)

    target_img_path = f"../hyperism/{scene_name}/images/scene_cam_00_final_preview/frame.0000.color.jpg"
    if not Path(target_img_path).exists():
        print("Skipping:", scene_name)
        continue

    score = inspector.generate_score(
        target_image_path=target_img_path,
        rendering_image_path=f"../outputs/{scene_name}.png",
        prompt=prompt
    )

    scores[scene_name] = score

    sleep(15)

Evaluating: ai_001_001
Evaluating: ai_001_008
Evaluating: ai_002_004
Evaluating: ai_002_005
Evaluating: ai_003_005
Evaluating: ai_003_008
Evaluating: ai_004_007
Evaluating: ai_005_010
Evaluating: ai_006_006
Evaluating: ai_007_005
Evaluating: ai_008_008
Evaluating: ai_009_001
Evaluating: ai_016_009
Evaluating: ai_017_001
Evaluating: ai_017_002
Evaluating: ai_017_004
Evaluating: ai_021_002
Evaluating: ai_023_006
Evaluating: ai_026_016
Evaluating: ai_027_009
Evaluating: ai_028_001
Evaluating: ai_031_010
Evaluating: ai_034_002
Evaluating: ai_044_009
Evaluating: ai_045_008
Evaluating: ai_045_010
Evaluating: ai_046_004
Evaluating: ai_050_005
Evaluating: ai_053_004
Skipping: ai_053_004
Evaluating: ai_055_004


In [13]:
import pickle

with open("../outputs/scores.pkl", "wb") as f:
    pickle.dump(scores, f)

In [11]:
import pandas as pd


records = []
for scene_name, score in scores.items():
    records.append({
        "scene_name" : scene_name,
        "object_count" : score.object_count,
        "object_orientation" : score.object_orientation,
        "object_scale" : score.object_scale,
        "object_position" : score.object_placement,
    })

score_df = pd.DataFrame.from_records(records)
score_df["avg_score"] = score_df[["object_count", "object_orientation", "object_scale", "object_position"]].mean(axis=1)

In [12]:
score_df

,scene_name,object_count,object_orientation,object_scale,object_position,avg_score
0,ai_001_001,80,90,70,85,81.25
1,ai_001_008,100,100,95,95,97.50
2,ai_002_004,20,20,20,20,20.00
3,ai_002_005,20,20,20,20,20.00
4,ai_003_005,50,30,30,40,37.50
5,ai_003_008,20,20,10,10,15.00
6,ai_004_007,80,40,30,40,47.50
7,ai_005_010,0,0,0,0,0.00
8,ai_006_006,20,20,20,20,20.00
9,ai_007_005,95,95,85,90,91.25
